# GSM State Function Subclasses - Thermodynamic Type Safety

This notebook demonstrates the use of specialized thermodynamic state function subclasses that provide explicit variable naming and type safety while maintaining the general framework capabilities.

The subclasses (`GSMHelmholtzFn`, `GSMGibbsFn`, `GSMEnthalpyFn`, `GSMInternalEnergyFn`) establish correct mappings between explicit variables (T, S, eps, sig) and the general framework (th_x, th_y, mc_x, mc_y).

In [ ]:
import sympy as sp
from bmcs_matmod.gsm_lagrange.core2 import gsm_vars
from bmcs_matmod.gsm_lagrange.core2.gsm_state_fn_old import (
    GSMStateFn, StateFunction,
    GSMHelmholtzFn, GSMGibbsFn, GSMEnthalpyFn, GSMInternalEnergyFn
)
from bmcs_matmod.gsm_lagrange.core2.gsm_thermodyn_box import GSMThermodynBox
from bmcs_matmod.gsm_lagrange.core2.gsm_thermodyn_box_widget import create_interactive_widget
from bmcs_matmod.gsm_lagrange.core2.gsm_vars import Scalar

## Create Common Variables

We'll define the standard thermodynamic and mechanical variables that will be used across all state function types.

In [2]:
# Standard thermodynamic variables
T = Scalar(r'\vartheta', codename='T', real=True, positive=True)    # Temperature
S = Scalar('S', codename='S', real=True)                            # Entropy

# Standard mechanical variables  
eps = Scalar(r'\varepsilon', codename='eps', real=True)             # Strain
sig = Scalar(r'\sigma', codename='sig', real=True)                  # Stress

# Internal variables (damage model)
omega = Scalar(r'\omega', codename='omega', real=True, positive=True)  # Damage variable
Y = Scalar('Y', codename='Y', real=True)                               # Stored energy variable

# Material parameters
E = Scalar('E', codename='E', positive=True)                        # Young's modulus
C_eps = Scalar(r'C_{\varepsilon}', codename='C_eps', positive=True) # Heat capacity at constant strain
T_0 = Scalar(r'\vartheta_0', codename='T_0', positive=True)         # Reference temperature

print("Variables defined:")
print(f"Thermal: T={T}, S={S}")
print(f"Mechanical: eps={eps}, sig={sig}")
print(f"Internal: omega={omega}, Y={Y}")
print(f"Parameters: E={E}, C_eps={C_eps}, T_0={T_0}")

Variables defined:
Thermal: T=\vartheta, S=S
Mechanical: eps=\varepsilon, sig=\sigma
Internal: omega=\omega, Y=Y
Parameters: E=E, C_eps=C_{\varepsilon}, T_0=\vartheta_0


## 1. Helmholtz Free Energy: F(T, ε, ω)

The Helmholtz free energy with temperature and strain as natural variables.
Natural: T, ε, ω | Conjugate: S, σ, Y

In [3]:
# Define Helmholtz free energy expression F(T, ε, ω)
F_elastic = sp.Rational(1, 2) * (1 - omega) * E * eps**2
F_thermal = C_eps * (T - T * sp.log(T / T_0))
F_expr = F_elastic + F_thermal

# Create Helmholtz state function using the specialized subclass
helmholtz_fn = GSMHelmholtzFn(
    fn_expr=F_expr,
    T=T,        # Temperature (natural)
    S=S,        # Entropy (conjugate)
    eps=eps,    # Strain (natural)
    sig=sig,    # Stress (conjugate)
    Eps=omega,  # Internal natural (damage)
    Sig=Y       # Internal conjugate (stored energy)
)

print("Helmholtz Free Energy State Function:")
from IPython.display import display, Markdown
display(Markdown(helmholtz_fn.markdown_overview()))

Helmholtz Free Energy State Function:


# GSM State Function Overview

$F = C_{\varepsilon} \left(- \vartheta \log{\left(\frac{\vartheta}{\vartheta_{0}} \right)} + \vartheta\right) + E \varepsilon^{2} \left(\frac{1}{2} - \frac{\omega}{2}\right)$

## Natural Variables (independent)
| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| None | $\vartheta$ | `T` | K | K | 0 | 1d | positive, nonnegative, real, finite |
| None | $\varepsilon$ | `eps` | 1 | 1 | 0 | 1d | real, finite |
| None | $\omega$ | `omega` | 1 | 1 | 0 | 1d | positive, nonnegative, real, finite |


## Conjugate Variables (derivatives)
| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| None | $S$ | `S` | J⋅K⁻¹ | J⋅K⁻¹ | 0 | 1d | real, finite |
| None | $\sigma$ | `sig` | Pa | Pa | 0 | 1d | real, finite |
| None | $Y$ | `Y` | J⋅m⁻³ | J⋅m⁻³ | 0 | 1d | real, finite |


## Constitutive relations
$$ S = \frac{\partial f}{\partial \vartheta} = - \log{\left(\left(\frac{\vartheta}{\vartheta_{0}}\right)^{C_{\varepsilon}} \right)} $$
$$ \sigma = \frac{\partial f}{\partial \varepsilon} = E \varepsilon \left(1 - \omega\right) $$
$$ Y = \frac{\partial f}{\partial \omega} = - \frac{E \varepsilon^{2}}{2} $$

## Expected Variable Organization
- **Natural:** ['T', 'eps', 'Eps']
- **Conjugate:** ['S', 'sig', 'Sig']
- **Thermally Intensive:** True
- **Mechanically Intensive:** False
- **Transformation Targets:** ['U', 'G', 'H']


## 2. Gibbs Free Energy: G(T, σ, ω)

The Gibbs free energy with temperature and stress as natural variables.
Natural: T, σ, ω | Conjugate: S, ε, Y

In [4]:
# Define Gibbs free energy expression G(T, σ, ω)
# G = F - ε*σ (Legendre transform w.r.t. mechanical variables)
G_elastic = -sp.Rational(1, 2) * (1 - omega) * sig**2 / E  # Transform: (1/2)*E*ε² → -(1/2)*σ²/E
G_thermal = C_eps * (T - T * sp.log(T / T_0))              # Same thermal part
G_expr = G_elastic + G_thermal

# Create Gibbs state function using the specialized subclass
gibbs_fn = GSMGibbsFn(
    fn_expr=G_expr,
    T=T,        # Temperature (natural)
    S=S,        # Entropy (conjugate)
    sig=sig,    # Stress (natural)
    eps=eps,    # Strain (conjugate)
    Eps=omega,  # Internal natural (damage)
    Sig=Y       # Internal conjugate (stored energy)
)

print("Gibbs Free Energy State Function:")
display(Markdown(gibbs_fn.markdown_overview()))

Gibbs Free Energy State Function:


# GSM State Function Overview

$G = C_{\varepsilon} \left(- \vartheta \log{\left(\frac{\vartheta}{\vartheta_{0}} \right)} + \vartheta\right) + \frac{\sigma^{2} \left(\frac{\omega}{2} - \frac{1}{2}\right)}{E}$

## Natural Variables (independent)
| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| None | $\vartheta$ | `T` | K | K | 0 | 1d | positive, nonnegative, real, finite |
| None | $\sigma$ | `sig` | Pa | Pa | 0 | 1d | real, finite |
| None | $\omega$ | `omega` | 1 | 1 | 0 | 1d | positive, nonnegative, real, finite |


## Conjugate Variables (derivatives)
| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| None | $S$ | `S` | J⋅K⁻¹ | J⋅K⁻¹ | 0 | 1d | real, finite |
| None | $\varepsilon$ | `eps` | 1 | 1 | 0 | 1d | real, finite |
| None | $Y$ | `Y` | J⋅m⁻³ | J⋅m⁻³ | 0 | 1d | real, finite |


## Constitutive relations
$$ S = \frac{\partial f}{\partial \vartheta} = - \log{\left(\left(\frac{\vartheta}{\vartheta_{0}}\right)^{C_{\varepsilon}} \right)} $$
$$ \varepsilon = \frac{\partial f}{\partial \sigma} = \frac{\sigma \left(\omega - 1\right)}{E} $$
$$ Y = \frac{\partial f}{\partial \omega} = \frac{\sigma^{2}}{2 E} $$

## Expected Variable Organization
- **Natural:** ['T', 'sig', 'Eps']
- **Conjugate:** ['S', 'eps', 'Sig']
- **Thermally Intensive:** True
- **Mechanically Intensive:** True
- **Transformation Targets:** ['F', 'H', 'U']


## 3. Enthalpy: H(S, σ, ω)

The enthalpy with entropy and stress as natural variables.
Natural: S, σ, ω | Conjugate: T, ε, Y

In [5]:
# Define enthalpy expression H(S, σ, ω)
# H = U + σ*ε, and we need to express in terms of (S, σ, ω)
# For simplicity, we'll use a model form that's consistent with the thermal-mechanical structure
H_elastic = -sp.Rational(1, 2) * (1 - omega) * sig**2 / E  # Mechanical part (same as Gibbs)
H_thermal = C_eps * T_0 * sp.exp(S / C_eps)                # Thermal part in terms of S
H_expr = H_elastic + H_thermal

# Create Enthalpy state function using the specialized subclass
enthalpy_fn = GSMEnthalpyFn(
    fn_expr=H_expr,
    S=S,        # Entropy (natural)
    T=T,        # Temperature (conjugate)
    sig=sig,    # Stress (natural)
    eps=eps,    # Strain (conjugate)
    Eps=omega,  # Internal natural (damage)
    Sig=Y       # Internal conjugate (stored energy)
)

print("Enthalpy State Function:")
display(Markdown(enthalpy_fn.markdown_overview()))

Enthalpy State Function:


# GSM State Function Overview

$H = C_{\varepsilon} \vartheta_{0} e^{\frac{S}{C_{\varepsilon}}} + \frac{\sigma^{2} \left(\frac{\omega}{2} - \frac{1}{2}\right)}{E}$

## Natural Variables (independent)
| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| None | $S$ | `S` | J⋅K⁻¹ | J⋅K⁻¹ | 0 | 1d | real, finite |
| None | $\sigma$ | `sig` | Pa | Pa | 0 | 1d | real, finite |
| None | $\omega$ | `omega` | 1 | 1 | 0 | 1d | positive, nonnegative, real, finite |


## Conjugate Variables (derivatives)
| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| None | $\vartheta$ | `T` | K | K | 0 | 1d | positive, nonnegative, real, finite |
| None | $\varepsilon$ | `eps` | 1 | 1 | 0 | 1d | real, finite |
| None | $Y$ | `Y` | J⋅m⁻³ | J⋅m⁻³ | 0 | 1d | real, finite |


## Constitutive relations
$$ \vartheta = \frac{\partial f}{\partial S} = \vartheta_{0} e^{\frac{S}{C_{\varepsilon}}} $$
$$ \varepsilon = \frac{\partial f}{\partial \sigma} = \frac{\sigma \left(\omega - 1\right)}{E} $$
$$ Y = \frac{\partial f}{\partial \omega} = \frac{\sigma^{2}}{2 E} $$

## Expected Variable Organization
- **Natural:** ['S', 'sig', 'Eps']
- **Conjugate:** ['T', 'eps', 'Sig']
- **Thermally Intensive:** False
- **Mechanically Intensive:** True
- **Transformation Targets:** ['U', 'G', 'F']


## 4. Internal Energy: U(S, ε, ω)

The internal energy with entropy and strain as natural variables.
Natural: S, ε, ω | Conjugate: T, σ, Y

In [6]:
# Define internal energy expression U(S, ε, ω)
U_elastic = sp.Rational(1, 2) * (1 - omega) * E * eps**2   # Mechanical part (same as Helmholtz)
U_thermal = C_eps * T_0 * sp.exp(S / C_eps)                # Thermal part in terms of S
U_expr = U_elastic + U_thermal

# Create Internal Energy state function using the specialized subclass
internal_energy_fn = GSMInternalEnergyFn(
    fn_expr=U_expr,
    S=S,        # Entropy (natural)
    T=T,        # Temperature (conjugate)
    eps=eps,    # Strain (natural)
    sig=sig,    # Stress (conjugate)
    Eps=omega,  # Internal natural (damage)
    Sig=Y       # Internal conjugate (stored energy)
)

print("Internal Energy State Function:")
display(Markdown(internal_energy_fn.markdown_overview()))

Internal Energy State Function:


# GSM State Function Overview

$U = C_{\varepsilon} \vartheta_{0} e^{\frac{S}{C_{\varepsilon}}} + E \varepsilon^{2} \left(\frac{1}{2} - \frac{\omega}{2}\right)$

## Natural Variables (independent)
| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| None | $S$ | `S` | J⋅K⁻¹ | J⋅K⁻¹ | 0 | 1d | real, finite |
| None | $\varepsilon$ | `eps` | 1 | 1 | 0 | 1d | real, finite |
| None | $\omega$ | `omega` | 1 | 1 | 0 | 1d | positive, nonnegative, real, finite |


## Conjugate Variables (derivatives)
| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| None | $\vartheta$ | `T` | K | K | 0 | 1d | positive, nonnegative, real, finite |
| None | $\sigma$ | `sig` | Pa | Pa | 0 | 1d | real, finite |
| None | $Y$ | `Y` | J⋅m⁻³ | J⋅m⁻³ | 0 | 1d | real, finite |


## Constitutive relations
$$ \vartheta = \frac{\partial f}{\partial S} = \vartheta_{0} e^{\frac{S}{C_{\varepsilon}}} $$
$$ \sigma = \frac{\partial f}{\partial \varepsilon} = E \varepsilon \left(1 - \omega\right) $$
$$ Y = \frac{\partial f}{\partial \omega} = - \frac{E \varepsilon^{2}}{2} $$

## Expected Variable Organization
- **Natural:** ['S', 'eps', 'Eps']
- **Conjugate:** ['T', 'sig', 'Sig']
- **Thermally Intensive:** False
- **Mechanically Intensive:** False
- **Transformation Targets:** ['F', 'H', 'G']


## Demonstrate Type Safety and Explicit Variables

Show how the subclasses provide explicit variable access while maintaining compatibility with the general framework.

In [7]:
# Demonstrate type safety through explicit variable access
print("Type Safety Example - Explicit Variable Access:")
print(f"Helmholtz: helmholtz_fn.T = {helmholtz_fn.T}, helmholtz_fn.eps = {helmholtz_fn.eps}")
print(f"Gibbs: gibbs_fn.T = {gibbs_fn.T}, gibbs_fn.sig = {gibbs_fn.sig}")

print("\nAll subclasses use the same general framework mapping:")
state_functions = [
    ("Helmholtz F(T,ε,ω)", helmholtz_fn),
    ("Gibbs G(T,σ,ω)", gibbs_fn), 
    ("Enthalpy H(S,σ,ω)", enthalpy_fn),
    ("Internal Energy U(S,ε,ω)", internal_energy_fn)
]

for name, fn in state_functions:
    print(f"{name}: th_x={fn.th_x_var}, mc_x={fn.mc_x_var} → {fn.state_function_type.value}")

Type Safety Example - Explicit Variable Access:
Helmholtz: helmholtz_fn.T = \vartheta, helmholtz_fn.eps = \varepsilon
Gibbs: gibbs_fn.T = \vartheta, gibbs_fn.sig = \sigma

All subclasses use the same general framework mapping:
Helmholtz F(T,ε,ω): th_x=\vartheta, mc_x=\varepsilon → F
Gibbs G(T,σ,ω): th_x=\vartheta, mc_x=\sigma → G
Enthalpy H(S,σ,ω): th_x=S, mc_x=\sigma → H
Internal Energy U(S,ε,ω): th_x=S, mc_x=\varepsilon → U


## Create Thermodynamic Box with Helmholtz Function

Now we'll create a thermodynamic box using the Helmholtz state function and demonstrate the interactive widget.

In [8]:
# Create the thermodynamic box using the Helmholtz state function
gsm_box = GSMThermodynBox(
    initial_state_fn=StateFunction.HELMHOLTZ,
    initial_state_instance=helmholtz_fn,
)

print("Thermodynamic Box created with Helmholtz state function")

Thermodynamic Box created with Helmholtz state function


## Create and Display Interactive Widget

Create the interactive widget to explore thermodynamic relationships using the typified state function.

In [9]:
# Create the interactive widget with the thermodynamic box
widget = create_interactive_widget(
    gsm_box=gsm_box,
    title="GSM Thermodynamic Box - State Function Subclasses Demo"
)

# Display the widget
widget.show()

## Test Multiple State Function Types in Widgets

Create thermodynamic boxes for different state function types and show them in separate widgets.

In [11]:
# Create boxes for all four state function types
state_functions_data = [
    ("Helmholtz F(T,ε,ω)", StateFunction.HELMHOLTZ, helmholtz_fn),
    ("Gibbs G(T,σ,ω)", StateFunction.GIBBS, gibbs_fn),
    ("Enthalpy H(S,σ,ω)", StateFunction.ENTHALPY, enthalpy_fn),
    ("Internal Energy U(S,ε,ω)", StateFunction.INTERNAL_ENERGY, internal_energy_fn)
]

widgets = []
for name, state_fn_type, state_fn_instance in state_functions_data:
    box = GSMThermodynBox(
        initial_state_fn=state_fn_type,
        initial_state_instance=state_fn_instance,
    )
    widget = create_interactive_widget(gsm_box=box, title=f"GSM Box - {name}")
    widgets.append((name, widget))

print(f"Created {len(widgets)} widgets for different state function types")

Created 4 widgets for different state function types


In [12]:
# Display all widgets to demonstrate universality across thermodynamic potentials
for name, widget in widgets:
    print(f"--- {name} ---")
    widget.show()

--- Helmholtz F(T,ε,ω) ---


--- Helmholtz F(T,ε,ω) ---


--- Helmholtz F(T,ε,ω) ---


--- Gibbs G(T,σ,ω) ---


--- Enthalpy H(S,σ,ω) ---


--- Internal Energy U(S,ε,ω) ---


## Summary and Key Benefits

This notebook demonstrated the thermodynamic state function subclasses and their key benefits:

### 1. Type Safety
- Explicit variable attributes (T, S, eps, sig) prevent thermodynamic errors
- Constructor enforces correct variable assignment for each potential type
- Clear distinction between natural and conjugate variables

### 2. Framework Generality  
- All subclasses work with the same derivation and execution machinery
- Universal thermodynamic box and widget system
- Consistent API across all four thermodynamic potentials

### 3. Code Reusability
- Same thermodynamic box handles all state function types
- Same widget system works for all potentials
- Same constitutive relation computation for all types

### 4. Extensibility
- Easy to add new thermodynamic potentials
- Framework extension requires only variable mapping
- Existing engines automatically work with new types

### 5. Maintainability
- Changes to base framework benefit all subclasses automatically
- Clear separation between thermodynamic specialization and general framework
- Type-safe variable access reduces debugging effort

This architecture establishes the foundation for the evolutionary material model framework described in `FROM_STATE_TO_EVOLUTION.md`.